# Breast Cancer Wisconsin - Neural Network Classification
## Chẩn đoán khối u vú: Lành tính (Benign) vs Ác tính (Malignant)

**Pipeline:** EDA → Tiền xử lý → Xây dựng MLP → Đánh giá → Thảo luận

In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_score, recall_score, f1_score)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print(f'Pandas version: {pd.__version__}')

# PHẦN 1: Khám phá & Trực quan hóa Dữ liệu (20%)

In [ ]:
# 1.1 Tải dữ liệu và kiểm tra missing values
df = pd.read_csv("archive (1)/data.csv")

# Xóa cột rỗng cuối (Unnamed: 32) và cột id không cần thiết
df.drop(columns=["Unnamed: 32", "id"], inplace=True, errors="ignore")

print("=" * 50)
print(f"Kích thước tập dữ liệu: {df.shape[0]} mẫu, {df.shape[1]} cột")
print(f"Số đặc trưng (features): {df.shape[1] - 1}")
print("=" * 50)
print("\nKiểm tra Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Không có giá trị khuyết thiếu!")
print(f"\nTổng missing: {df.isnull().sum().sum()}")
df.head()

In [ ]:
# 1.2 Thống kê mô tả & phân bố cột mục tiêu
print("Phân bố cột diagnosis:")
print(df["diagnosis"].value_counts())
print(f'\nTỷ lệ B: {df["diagnosis"].value_counts(normalize=True)["B"]:.2%}')
print(f'Tỷ lệ M: {df["diagnosis"].value_counts(normalize=True)["M"]:.2%}')
print("\nThống kê mô tả:")
df.describe().round(3)

In [ ]:
# 1.3 Bar chart & Pie chart - Tỷ lệ giữa 2 lớp (Class Imbalance)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#2ecc71", "#e74c3c"]
counts = df["diagnosis"].value_counts()

# Bar chart
axes[0].bar(counts.index, counts.values, color=colors, edgecolor="black")
axes[0].set_title("Phân bố lớp (Bar Chart)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Diagnosis")
axes[0].set_ylabel("Số lượng")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(counts.values, labels=["Benign (B)", "Malignant (M)"],
           colors=colors, autopct="%1.1f%%", startangle=90,
           explode=(0.05, 0.05), shadow=True)
axes[1].set_title("Tỷ lệ lớp (Pie Chart)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# 1.4 Histogram/Density plot - Phân bố đặc trưng theo 2 lớp
# Chọn 5 đặc trưng quan trọng
features_to_plot = ["radius_mean", "texture_mean", "area_mean",
                    "concavity_mean", "concave points_mean"]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, feat in enumerate(features_to_plot):
    for label, color in zip(["B", "M"], ["#2ecc71", "#e74c3c"]):
        subset = df[df["diagnosis"] == label][feat]
        axes[i].hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
        subset.plot.kde(ax=axes[i], color=color, linewidth=2)
    axes[i].set_title(feat, fontsize=12, fontweight="bold")
    axes[i].legend()

plt.suptitle("Phân bố đặc trưng theo lớp (B vs M)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("=> Nhận xét: area_mean, concavity_mean, concave points_mean phân tách 2 lớp tốt nhất.")

In [ ]:
# 1.5 Heatmap tương quan (Correlation)
plt.figure(figsize=(20, 16))

# Encode diagnosis tạm để tính correlation
df_corr = df.copy()
df_corr["diagnosis"] = df_corr["diagnosis"].map({"M": 1, "B": 0})

corr_matrix = df_corr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, linewidths=0.5,
            annot_kws={"size": 7})
plt.title("Ma trận tương quan giữa các đặc trưng", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Top features tương quan với diagnosis
print("\nTop 10 đặc trưng tương quan mạnh nhất với diagnosis:")
target_corr = corr_matrix["diagnosis"].drop("diagnosis").abs().sort_values(ascending=False)
print(target_corr.head(10).round(4))

print("\nCặp đặc trưng có multicollinearity cao (|corr| > 0.9):")
high_corr = []
cols = corr_matrix.columns.drop("diagnosis")
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        if abs(corr_matrix.loc[cols[i], cols[j]]) > 0.9:
            high_corr.append((cols[i], cols[j], round(corr_matrix.loc[cols[i], cols[j]], 3)))
for a, b, c in high_corr:
    print(f"  {a} <-> {b}: {c}")

# PHẦN 2: Tiền xử lý Dữ liệu (20%)

In [ ]:
# 2.1 Label Encoding: M -> 1 (Ác tính), B -> 0 (Lành tính)
le = LabelEncoder()
df["diagnosis"] = le.fit_transform(df["diagnosis"])  # B=0, M=1

print("Mapping: B (Benign) -> 0, M (Malignant) -> 1")
print(f"Phân bố sau encoding:\n{df['diagnosis'].value_counts()}")

# Tách features (X) và target (y)
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# 2.2 Tách dữ liệu: Train (70%) / Validation (15%) / Test (15%)
# Dùng stratify để đảm bảo phân phối lớp đồng đều

# Bước 1: Tách Train (70%) vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Bước 2: Tách Temp thành Validation (15%) và Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train set: {X_train.shape[0]} mẫu ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val set:   {X_val.shape[0]} mẫu ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} mẫu ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nPhân bố lớp trong Train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Phân bố lớp trong Val:   {dict(zip(*np.unique(y_val, return_counts=True)))}")
print(f"Phân bố lớp trong Test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

### Tại sao mạng nơ-ron nhạy cảm với scale của dữ liệu?

**Lý do cần chuẩn hóa dữ liệu cho Neural Network:**

1. **Gradient Descent hội tụ chậm**: Khi các đặc trưng có scale khác nhau lớn (ví dụ: `area_mean` ~ 100-2500 vs `smoothness_mean` ~ 0.05-0.16), gradient của các trọng số sẽ chênh lệch rất lớn. Điều này khiến quá trình tối ưu dao động mạnh (zigzag) và cần rất nhiều epoch để hội tụ.

2. **Hàm kích hoạt bị bão hòa**: Với sigmoid/tanh, nếu đầu vào có giá trị rất lớn, neuron sẽ rơi vào vùng bão hòa → gradient gần bằng 0 (vanishing gradient) → mạng không học được.

3. **Trọng số khởi tạo**: Các phương pháp khởi tạo trọng số (Xavier, He) giả định dữ liệu đầu vào có phân phối chuẩn hóa. Nếu dữ liệu chưa scale, việc khởi tạo sẽ không tối ưu.

**Tại sao chỉ được fit scaler trên tập Train?**

- Nếu fit scaler trên toàn bộ dữ liệu (bao gồm cả Val/Test), thông tin thống kê (mean, std) của tập Test sẽ "rò rỉ" vào quá trình tiền xử lý → **Data Leakage**.
- Trong thực tế, khi mô hình được triển khai, ta không có dữ liệu test/tương lai. Do đó, scaler chỉ được fit trên dữ liệu huấn luyện, rồi `transform` cho Val/Test để mô phỏng đúng điều kiện thực tế.

In [ ]:
# 2.3 Feature Scaling - StandardScaler
# CHỈ fit trên tập Train, rồi transform cho Val và Test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform trên Train
X_val_scaled = scaler.transform(X_val)          # chỉ transform trên Val
X_test_scaled = scaler.transform(X_test)        # chỉ transform trên Test

print("Trước khi scale (Train):")
print(f"  Mean: {X_train.values.mean():.4f}, Std: {X_train.values.std():.4f}")
print(f"  Min:  {X_train.values.min():.4f}, Max: {X_train.values.max():.4f}")

print("\nSau khi scale (Train):")
print(f"  Mean: {X_train_scaled.mean():.4f}, Std: {X_train_scaled.std():.4f}")
print(f"  Min:  {X_train_scaled.min():.4f}, Max: {X_train_scaled.max():.4f}")

# PHẦN 3: Xây dựng và Huấn luyện Mô hình với TensorFlow (30%)

In [ ]:
# 3.1 Kiến trúc mạng MLP (Multi-Layer Perceptron)
# - Sử dụng Keras Sequential API
# - 4 Hidden layers với Dropout + L2 Regularization để chống overfitting
# - Output layer: sigmoid (trả về xác suất 0-1 cho phân loại nhị phân)

n_features = X_train_scaled.shape[1]

model = keras.Sequential([
    # Input layer
    layers.Input(shape=(n_features,)),
    
    # Hidden layer 1: 128 neurons, ReLU, L2 regularization
    layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden layer 2: 64 neurons
    layers.Dense(64, activation="relu",
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden layer 3: 32 neurons
    layers.Dense(32, activation="relu",
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    
    # Hidden layer 4: 16 neurons
    layers.Dense(16, activation="relu",
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    
    # Output layer: 1 neuron, sigmoid -> xác suất [0, 1]
    # Sigmoid được chọn vì bài toán binary classification:
    #   output > 0.5 -> Malignant (1)
    #   output <= 0.5 -> Benign (0)
    layers.Dense(1, activation="sigmoid")
])

model.summary()

In [ ]:
# 3.2 Compile mô hình
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",  # Hàm mất mát cho phân loại nhị phân
    metrics=["accuracy"]
)

# 3.3 Kỹ thuật chống Overfitting: Early Stopping
# Dừng training nếu val_loss không cải thiện sau 15 epochs
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

print("Model compiled!")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Loss: Binary Crossentropy")
print(f"Metrics: Accuracy")
print(f"Anti-overfitting: Dropout + L2 Regularization + Early Stopping + BatchNorm")

In [ ]:
# 3.4 Huấn luyện mô hình
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print(f"\nTraining hoàn tất sau {len(history.history['loss'])} epochs")

# PHẦN 4: Đánh giá & Trực quan hóa Kết quả Mô hình (20%)

In [ ]:
# 4.1 Đồ thị Loss và Accuracy - Train vs Validation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"], label="Train Loss", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Val Loss", linewidth=2)
axes[0].set_title("Loss qua các Epoch", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history["accuracy"], label="Train Accuracy", linewidth=2)
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy", linewidth=2)
axes[1].set_title("Accuracy qua các Epoch", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=> Nếu Train Loss giảm nhưng Val Loss tăng → Overfitting.")
print("=> Nếu cả 2 đường Loss gần nhau và cùng giảm → Mô hình generalize tốt.")

In [ ]:
# 4.2 Dự đoán trên tập Test
y_pred_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

# Đánh giá trên tập Test
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

### Giải thích các khái niệm trong ngữ cảnh Y tế (Chẩn đoán ung thư vú):

| Khái niệm | Ý nghĩa | Ngữ cảnh Y tế |
|---|---|---|
| **True Positive (TP)** | Dự đoán Malignant → Thực tế Malignant | Phát hiện đúng bệnh nhân ung thư → Được điều trị kịp thời ✅ |
| **True Negative (TN)** | Dự đoán Benign → Thực tế Benign | Xác nhận đúng người khỏe mạnh → Tránh điều trị không cần thiết ✅ |
| **False Positive (FP)** | Dự đoán Malignant → Thực tế Benign | Chẩn đoán nhầm người khỏe thành ung thư → Gây lo lắng, xét nghiệm thêm ⚠️ |
| **False Negative (FN)** | Dự đoán Benign → Thực tế Malignant | **BỎ SÓT ung thư** → Bệnh nhân không được điều trị → Nguy hiểm tính mạng ❌ |

In [ ]:
# 4.3 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign (0)", "Malignant (1)"],
            yticklabels=["Benign (0)", "Malignant (1)"],
            annot_kws={"size": 16})
plt.title("Ma trận nhầm lẫn (Confusion Matrix)", fontsize=14, fontweight="bold")
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negative  (TN): {tn} - Đúng Benign")
print(f"False Positive (FP): {fp} - Nhầm Benign thành Malignant")
print(f"False Negative (FN): {fn} - BỎ SÓT ung thư (nguy hiểm!)")
print(f"True Positive  (TP): {tp} - Đúng Malignant")

In [ ]:
# 4.4 Precision, Recall, F1-Score
print("=" * 50)
print("Classification Report:")
print("=" * 50)
print(classification_report(y_test, y_pred,
                           target_names=["Benign (0)", "Malignant (1)"]))

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.4f}")
print(f"  → Trong các ca dự đoán Malignant, {precision*100:.1f}% thực sự là Malignant")
print(f"\nRecall: {recall:.4f}")
print(f"  → Trong các ca thực sự Malignant, mô hình phát hiện được {recall*100:.1f}%")
print(f"\nF1-Score: {f1:.4f}")
print(f"  → Trung bình điều hòa giữa Precision và Recall")

In [ ]:
# 4.5 ROC Curve và AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="#e74c3c", linewidth=2,
         label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1,
         label="Random Classifier")
plt.fill_between(fpr, tpr, alpha=0.1, color="#e74c3c")
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve", fontsize=14, fontweight="bold")
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC = {roc_auc:.4f}")
print("=> AUC càng gần 1.0, mô hình phân tách 2 lớp càng tốt.")
print("=> AUC = 0.5 tương đương đoán ngẫu nhiên.")

# PHẦN 5: Câu hỏi Tư duy / Thảo luận (10%)

## Câu 1: False Negative vs False Positive - Trường hợp nào nghiêm trọng hơn?

**Trả lời:** Trong bài toán chẩn đoán ung thư vú, **False Negative (FN) gây hậu quả nghiêm trọng hơn** rất nhiều so với False Positive (FP).

**Phân tích chi tiết:**

| | False Negative (FN) | False Positive (FP) |
|---|---|---|
| **Ý nghĩa** | Bệnh nhân **bị ung thư** nhưng mô hình dự đoán **khỏe mạnh** | Người **khỏe mạnh** nhưng mô hình dự đoán **bị ung thư** |
| **Hậu quả** | Bệnh nhân **không được điều trị**, ung thư phát triển → **đe dọa tính mạng** | Bệnh nhân phải **xét nghiệm thêm**, gây lo lắng tâm lý, tốn chi phí |
| **Mức độ** | ❌ **Nguy hiểm chết người** | ⚠️ Phiền toái nhưng có thể khắc phục |

**Kết luận:** Ta nên **tối ưu Recall** (Sensitivity/True Positive Rate).

**Lý do:**
- **Recall = TP / (TP + FN)** → Recall cao nghĩa là giảm thiểu FN → ít bỏ sót bệnh nhân ung thư.
- Trong y tế, nguyên tắc "thà báo động nhầm còn hơn bỏ sót" luôn được ưu tiên.
- Nếu Recall = 100%, mọi bệnh nhân ung thư đều được phát hiện, dù có thể một số người khỏe bị chẩn đoán nhầm (FP cao hơn).
- Các ca FP có thể được xác nhận lại bằng sinh thiết (biopsy), nhưng FN thì bệnh nhân sẽ không biết mình bị bệnh.

## Câu 2: Nếu bỏ qua Feature Scaling, loss sẽ biến thiên như thế nào?

**Trả lời:** Nếu **không chuẩn hóa dữ liệu**, loss sẽ có các biểu hiện sau:

1. **Loss khởi đầu rất cao**: Do các đặc trưng có scale chênh lệch lớn (ví dụ: `area_mean` ∈ [100, 2500] vs `smoothness_mean` ∈ [0.05, 0.16]), gradient sẽ rất lớn và không ổn định.

2. **Loss dao động mạnh (oscillate)**: Gradient descent sẽ "zigzag" thay vì đi thẳng đến minimum, vì learning rate phù hợp với feature nhỏ sẽ quá lớn cho feature lớn và ngược lại.

3. **Hội tụ chậm hoặc không hội tụ**: Mô hình cần nhiều epoch hơn đáng kể để đạt được kết quả tương đương, hoặc thậm chí không thể hội tụ.

4. **Accuracy thấp hơn**: Mô hình cuối cùng sẽ có performance kém hơn so với mô hình được train trên dữ liệu đã scale.

**Minh chứng bằng code ở cell tiếp theo:**

In [ ]:
# Minh chứng: Train mô hình KHÔNG có Feature Scaling
model_no_scale = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
])

model_no_scale.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train trên dữ liệu CHƯA scale (X_train gốc)
history_no_scale = model_no_scale.fit(
    X_train.values, y_train,
    validation_data=(X_val.values, y_val),
    epochs=100,
    batch_size=32,
    verbose=0
)

print("Training xong model KHÔNG scale!")

In [ ]:
# So sánh Loss giữa 2 mô hình: có Scale vs không Scale
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
axes[0].plot(history.history["loss"], label="Train (Scaled)", linewidth=2, color="#2ecc71")
axes[0].plot(history.history["val_loss"], label="Val (Scaled)", linewidth=2, color="#27ae60", linestyle="--")
axes[0].plot(history_no_scale.history["loss"], label="Train (No Scale)", linewidth=2, color="#e74c3c")
axes[0].plot(history_no_scale.history["val_loss"], label="Val (No Scale)", linewidth=2, color="#c0392b", linestyle="--")
axes[0].set_title("So sánh Loss: Scaled vs No Scale", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(history.history["accuracy"], label="Train (Scaled)", linewidth=2, color="#2ecc71")
axes[1].plot(history.history["val_accuracy"], label="Val (Scaled)", linewidth=2, color="#27ae60", linestyle="--")
axes[1].plot(history_no_scale.history["accuracy"], label="Train (No Scale)", linewidth=2, color="#e74c3c")
axes[1].plot(history_no_scale.history["val_accuracy"], label="Val (No Scale)", linewidth=2, color="#c0392b", linestyle="--")
axes[1].set_title("So sánh Accuracy: Scaled vs No Scale", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# So sánh kết quả cuối
_, acc_scaled = model.evaluate(X_test_scaled, y_test, verbose=0)
_, acc_no_scale = model_no_scale.evaluate(X_test.values, y_test, verbose=0)
print(f"\nTest Accuracy (Scaled):    {acc_scaled:.4f}")
print(f"Test Accuracy (No Scale): {acc_no_scale:.4f}")
print(f"\n=> Mô hình có Scale cho kết quả tốt hơn và loss ổn định hơn!")